# COVID-19 Global Analysis — EDA, Trends, and Predictive Modeling
### A Comprehensive Data-Driven Exploration of the Pandemic's Global Impact

---

**Author:** Hassan Ali  
**Role:** Data Scientist & Machine Learning Engineer  
**LinkedIn:** [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)  
**GitHub:** [github.com/hassan-ali786](https://github.com/hassan-ali786)  
**Portfolio:** [hassanali-portfolio.vercel.app](https://hassanali-portfolio.vercel.app/)

---

## About This Notebook

The COVID-19 pandemic was the defining global health crisis of the 21st century. Between January 2020 and mid-2023, the SARS-CoV-2 virus infected over 700 million people and caused more than 7 million confirmed deaths worldwide — with true excess mortality estimates substantially higher.

This notebook performs a comprehensive analysis of the global COVID-19 dataset to answer questions that are both historically important and analytically rich:

- How did confirmed cases and deaths evolve over time globally and by country?
- Which countries were most severely impacted in absolute and per-capita terms?
- How did case fatality rates differ across countries and over time?
- What patterns emerged in wave structures across different regions?
- Can we model cumulative case trajectories using regression?
- What does the data reveal about pandemic management effectiveness?

---

## Table of Contents

1. Environment Setup
2. Data Loading and Inspection
3. Data Cleaning and Preprocessing
4. Global Overview
5. Time Series Analysis
6. Country-Level Analysis
7. Case Fatality Rate Analysis
8. Continental Analysis
9. Recovery Analysis
10. Predictive Modeling
11. Key Insights and Conclusions

---
*If this analysis adds value to your work, an upvote is appreciated.*

---
## Section 1: Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

BLUE   = '#1f77b4'
RED    = '#d62728'
GREEN  = '#2ca02c'
ORANGE = '#ff7f0e'

plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'font.family'       : 'serif'
})
sns.set_style('whitegrid')

print('Environment configured.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

---
## Section 2: Data Loading and Inspection

We use the COVID-19 dataset from Kaggle which contains country-wise daily records of confirmed cases, deaths, and recoveries. The data originates from Johns Hopkins University CSSE — the most widely cited COVID-19 data source during the pandemic.

In [ ]:
import os

# Auto-detect dataset path
possible_paths = [
    '/kaggle/input/corona-virus-report',
    '/kaggle/input/novel-corona-virus-2019-dataset',
    '/kaggle/input/covid19-global-forecasting-week-1',
    '/kaggle/input/covid-19-data'
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        print(f'Dataset found: {path}')
        print('Files available:')
        for f in sorted(os.listdir(path)):
            print(f'  {f}')
        break

if data_path is None:
    print('Please add a COVID-19 dataset from Kaggle.')
    print('Recommended: "Corona Virus Report" by imdevskp')

In [ ]:
# Load main dataset
file_priority = [
    'covid_19_clean_complete.csv',
    'country_wise_latest.csv',
    'worldometer_data.csv',
    'full_grouped.csv'
]

df = None
loaded_file = None
for fname in file_priority:
    fpath = os.path.join(data_path, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        loaded_file = fname
        print(f'Loaded: {fname}')
        break

print(f'Shape   : {df.shape}')
print(f'Columns : {df.columns.tolist()}')
df.head()

In [ ]:
# Missing values and data types
print('Missing Values:')
print(df.isnull().sum())
print()
print('Data Types:')
print(df.dtypes)

---
## Section 3: Data Cleaning and Preprocessing

COVID-19 datasets require careful preprocessing — column names differ across sources, date formats vary, and some entries contain negative values due to retroactive data corrections by health authorities.

In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('-', '_')
)

# Auto-detect key columns
col_map = {}
for col in df.columns:
    if any(k in col for k in ['confirmed', 'cases', 'total_cases']): col_map['confirmed'] = col
    if any(k in col for k in ['death', 'total_deaths']):             col_map['deaths']    = col
    if any(k in col for k in ['recover', 'total_recover']):          col_map['recovered'] = col
    if any(k in col for k in ['country', 'region', 'location']):     col_map['country']   = col
    if 'date' in col:                                                 col_map['date']      = col
    if 'continent' in col:                                            col_map['continent'] = col

# Only rename if not already correct
for std_name, orig_name in col_map.items():
    if orig_name != std_name and orig_name in df.columns:
        df.rename(columns={orig_name: std_name}, inplace=True)

print('Key columns identified and standardized:')
for k in ['country', 'date', 'confirmed', 'deaths', 'recovered', 'continent']:
    status = 'FOUND' if k in df.columns else 'NOT FOUND'
    print(f'  {k:<12}: {status}')

# Parse date
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Convert numerics and fill NaN
for col in ['confirmed', 'deaths', 'recovered']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)

# Derived columns
if 'confirmed' in df.columns and 'deaths' in df.columns:
    df['cfr'] = np.where(df['confirmed'] > 0,
                         (df['deaths'] / df['confirmed'] * 100).round(3), 0)

if all(c in df.columns for c in ['confirmed', 'deaths', 'recovered']):
    df['active'] = (df['confirmed'] - df['deaths'] - df['recovered']).clip(lower=0)

print(f'\nCleaning complete. Shape: {df.shape}')

---
## Section 4: Global Overview

We begin with the highest-level picture — total confirmed cases, deaths, and recoveries worldwide. These numbers represent the largest coordinated data collection effort in public health history.

In [ ]:
# Get latest snapshot
if 'date' in df.columns:
    latest = df[df['date'] == df['date'].max()]
else:
    latest = df.copy()

if 'country' in latest.columns:
    agg_cols = [c for c in ['confirmed', 'deaths', 'recovered'] if c in latest.columns]
    country_df = latest.groupby('country')[agg_cols].sum()
    country_df = country_df[country_df['confirmed'] > 0].sort_values('confirmed', ascending=False)
else:
    country_df = latest.copy()

total_cases     = country_df['confirmed'].sum()
total_deaths    = country_df['deaths'].sum()
total_recovered = country_df['recovered'].sum() if 'recovered' in country_df.columns else 0
global_cfr      = total_deaths / total_cases * 100 if total_cases > 0 else 0

print('--- Global COVID-19 Summary ---')
print(f'  Total Confirmed Cases : {total_cases:>15,.0f}')
print(f'  Total Deaths          : {total_deaths:>15,.0f}')
print(f'  Total Recovered       : {total_recovered:>15,.0f}')
print(f'  Global CFR            : {global_cfr:>14.2f}%')
print(f'  Countries in Dataset  : {len(country_df):>15,}')

In [ ]:
# Global Summary Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
metrics = ['Confirmed', 'Deaths', 'Recovered']
values  = [total_cases, total_deaths, total_recovered]
colors  = [BLUE, RED, GREEN]
bars = axes[0].bar(metrics, values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Global COVID-19 Totals')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f'{val/1e6:.1f}M', ha='center', fontweight='bold', fontsize=10)

# Outcome donut
if total_recovered > 0:
    active = max(total_cases - total_deaths - total_recovered, 0)
    sizes  = [total_recovered, total_deaths, active]
    labels = ['Recovered', 'Deaths', 'Active/Unknown']
    clrs   = [GREEN, RED, ORANGE]
else:
    sizes  = [total_cases - total_deaths, total_deaths]
    labels = ['Survived', 'Deaths']
    clrs   = [GREEN, RED]

axes[1].pie(sizes, labels=labels, colors=clrs, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(width=0.6))
axes[1].set_title('Global Outcome Distribution')

fig.suptitle('Global COVID-19 Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Interpretation:')
print(f'Global CFR of {global_cfr:.2f}% represents confirmed deaths per confirmed case.')
print(f'True infection fatality rate is substantially lower due to massive undercounting.')

---
## Section 5: Time Series Analysis

The pandemic evolved in distinct waves — periods of accelerating transmission separated by relative suppression. Each wave was driven by a dominant variant: Alpha, Delta, and Omicron being the most consequential.

In [ ]:
if 'date' in df.columns:
    global_ts = df.groupby('date')[['confirmed', 'deaths']].sum().reset_index().sort_values('date')
    global_ts['new_cases']    = global_ts['confirmed'].diff().clip(lower=0)
    global_ts['new_deaths']   = global_ts['deaths'].diff().clip(lower=0)
    global_ts['new_cases_7d'] = global_ts['new_cases'].rolling(7).mean()

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    # Cumulative
    axes[0].fill_between(global_ts['date'], global_ts['confirmed'], alpha=0.3, color=BLUE)
    axes[0].plot(global_ts['date'], global_ts['confirmed'], color=BLUE, linewidth=2, label='Confirmed Cases')
    axes[0].fill_between(global_ts['date'], global_ts['deaths'], alpha=0.6, color=RED)
    axes[0].plot(global_ts['date'], global_ts['deaths'], color=RED, linewidth=2, label='Deaths')
    axes[0].set_title('Cumulative Global COVID-19 Cases and Deaths')
    axes[0].set_ylabel('Cumulative Count')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
    axes[0].legend(fontsize=10)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

    # Daily new cases
    axes[1].bar(global_ts['date'], global_ts['new_cases'], color=BLUE, alpha=0.3, width=1)
    axes[1].plot(global_ts['date'], global_ts['new_cases_7d'], color=BLUE, linewidth=2.5,
                 label='7-day rolling average')
    axes[1].set_title('Daily New COVID-19 Cases — Global')
    axes[1].set_ylabel('Daily New Cases')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
    axes[1].legend(fontsize=10)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

    fig.suptitle('COVID-19 Global Time Series — Full Pandemic Period', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    peak_idx  = global_ts['new_cases_7d'].idxmax()
    peak_date = global_ts.loc[peak_idx, 'date']
    peak_val  = global_ts.loc[peak_idx, 'new_cases_7d']
    print(f'Peak daily cases (7-day avg): {peak_val:,.0f} on {peak_date.strftime("%B %d, %Y")}')
    print('The multiple peaks in the daily case chart correspond to distinct pandemic waves.')
    print('The Omicron wave (late 2021 / early 2022) produced the highest case counts globally.')
else:
    print('Date column not found — time series analysis skipped.')

---
## Section 6: Country-Level Analysis

Absolute case counts favor populous nations. We examine both absolute totals and relative rankings to understand which countries experienced the most severe outbreaks.

In [ ]:
top15_cases  = country_df.nlargest(15, 'confirmed')
top15_deaths = country_df.nlargest(15, 'deaths')

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

axes[0].barh(top15_cases.index[::-1], top15_cases['confirmed'][::-1], color=BLUE, edgecolor='white')
axes[0].set_title('Top 15 Countries — Total Confirmed Cases')
axes[0].set_xlabel('Confirmed Cases')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
for i, val in enumerate(top15_cases['confirmed'][::-1]):
    axes[0].text(val * 1.01, i, f'{val/1e6:.1f}M', va='center', fontsize=8.5)

axes[1].barh(top15_deaths.index[::-1], top15_deaths['deaths'][::-1], color=RED, edgecolor='white')
axes[1].set_title('Top 15 Countries — Total Deaths')
axes[1].set_xlabel('Deaths')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
for i, val in enumerate(top15_deaths['deaths'][::-1]):
    axes[1].text(val * 1.01, i, f'{val/1e3:.0f}K', va='center', fontsize=8.5)

fig.suptitle('Country-Level COVID-19 Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observation:')
print('The United States, India, and Brazil dominate by absolute case counts.')
print('Per-capita analysis (not shown here due to population data requirements)')
print('would reveal smaller nations with proportionally higher outbreak intensity.')

---
## Section 7: Case Fatality Rate Analysis

The Case Fatality Rate (CFR) is deaths divided by confirmed cases. It is one of the most debated metrics of the pandemic — varying 50x across countries due to testing capacity, healthcare quality, and population demographics.

In [ ]:
cfr_df = country_df[country_df['confirmed'] > 10000].copy()
cfr_df['cfr'] = (cfr_df['deaths'] / cfr_df['confirmed'] * 100).round(3)
cfr_df = cfr_df[cfr_df['cfr'] > 0].sort_values('cfr', ascending=False)

top15_cfr    = cfr_df.head(15)
bottom15_cfr = cfr_df.tail(15).sort_values('cfr')

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

axes[0].barh(top15_cfr.index[::-1], top15_cfr['cfr'][::-1], color=RED, edgecolor='white')
axes[0].set_title('Highest Case Fatality Rate by Country')
axes[0].set_xlabel('CFR (%)')
for i, val in enumerate(top15_cfr['cfr'][::-1]):
    axes[0].text(val + 0.02, i, f'{val:.2f}%', va='center', fontsize=9)

axes[1].barh(bottom15_cfr.index[::-1], bottom15_cfr['cfr'][::-1], color=GREEN, edgecolor='white')
axes[1].set_title('Lowest Case Fatality Rate by Country')
axes[1].set_xlabel('CFR (%)')
for i, val in enumerate(bottom15_cfr['cfr'][::-1]):
    axes[1].text(val + 0.001, i, f'{val:.3f}%', va='center', fontsize=9)

fig.suptitle('Case Fatality Rate Analysis by Country', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Global CFR: {(country_df["deaths"].sum() / country_df["confirmed"].sum() * 100):.3f}%')
print()
print('High CFR countries often reflect: limited testing, overwhelmed healthcare,')
print('older populations, or late pandemic response.')
print('Low CFR countries often reflect: mass testing, younger populations,')
print('effective healthcare, or early containment strategies.')

---
## Section 8: Continental Analysis

Continent-level patterns reveal how shared economic and institutional characteristics shaped regional pandemic outcomes.

In [ ]:
CONTINENT_COLORS = {
    'Asia': BLUE, 'Europe': ORANGE, 'North America': GREEN,
    'South America': RED, 'Africa': PURPLE if 'PURPLE' in dir() else '#9467bd', 'Oceania': '#8c564b'
}

if 'continent' in df.columns:
    if 'date' in df.columns:
        cont_df = df[df['date'] == df['date'].max()].groupby('continent')[['confirmed','deaths']].sum()
    else:
        cont_df = df.groupby('continent')[['confirmed','deaths']].sum()

    cont_df = cont_df[cont_df['confirmed'] > 0].sort_values('confirmed', ascending=False)
    cont_df['cfr'] = (cont_df['deaths'] / cont_df['confirmed'] * 100).round(3)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    colors_c = [CONTINENT_COLORS.get(c, BLUE) for c in cont_df.index]

    axes[0].bar(cont_df.index, cont_df['confirmed'], color=colors_c, edgecolor='white')
    axes[0].set_title('Total Cases by Continent')
    axes[0].set_ylabel('Confirmed Cases')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
    axes[0].set_xticklabels(cont_df.index, rotation=25, ha='right')

    axes[1].bar(cont_df.index, cont_df['deaths'], color=colors_c, edgecolor='white')
    axes[1].set_title('Total Deaths by Continent')
    axes[1].set_ylabel('Deaths')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
    axes[1].set_xticklabels(cont_df.index, rotation=25, ha='right')

    axes[2].bar(cont_df.index, cont_df['cfr'], color=colors_c, edgecolor='white')
    axes[2].set_title('CFR by Continent')
    axes[2].set_ylabel('CFR (%)')
    axes[2].set_xticklabels(cont_df.index, rotation=25, ha='right')
    for i, val in enumerate(cont_df['cfr']):
        axes[2].text(i, val + 0.01, f'{val:.2f}%', ha='center', fontsize=9)

    fig.suptitle('Continental COVID-19 Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(cont_df.to_string())
else:
    print('Continent column not found in this dataset.')

---
## Section 9: Recovery Analysis

Recovery rates complement CFR as a measure of healthcare system effectiveness. Countries with high testing and strong healthcare infrastructure tend to show higher recovery rates.

In [ ]:
if 'recovered' in country_df.columns and country_df['recovered'].sum() > 0:
    rec_df = country_df[country_df['confirmed'] > 10000].copy()
    rec_df['recovery_rate'] = (rec_df['recovered'] / rec_df['confirmed'] * 100).round(2)
    rec_df = rec_df[rec_df['recovery_rate'] > 0].sort_values('recovery_rate', ascending=False)

    top15_rec = rec_df.head(15)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].barh(top15_rec.index[::-1], top15_rec['recovery_rate'][::-1],
                 color=GREEN, edgecolor='white')
    axes[0].set_title('Top 15 Countries — Highest Recovery Rate')
    axes[0].set_xlabel('Recovery Rate (%)')
    for i, val in enumerate(top15_rec['recovery_rate'][::-1]):
        axes[0].text(val + 0.1, i, f'{val:.1f}%', va='center', fontsize=9)

    # CFR vs Recovery Rate
    if 'cfr' not in rec_df.columns:
        rec_df['cfr'] = rec_df['deaths'] / rec_df['confirmed'] * 100
    rec_scatter = rec_df[rec_df['cfr'] > 0]
    axes[1].scatter(rec_scatter['recovery_rate'], rec_scatter['cfr'],
                    alpha=0.6, color=BLUE, s=30, edgecolors='none')
    axes[1].set_title('Recovery Rate vs Case Fatality Rate')
    axes[1].set_xlabel('Recovery Rate (%)')
    axes[1].set_ylabel('CFR (%)')
    corr = rec_scatter['recovery_rate'].corr(rec_scatter['cfr'])
    axes[1].annotate(f'r = {corr:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                     fontsize=11, bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

    fig.suptitle('COVID-19 Recovery Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Recovery data not available or zero in this dataset.')
    print('Note: Many datasets discontinued recovery tracking after 2021.')

---
## Section 10: Predictive Modeling

We fit regression models to the cumulative global case time series. This demonstrates how polynomial and ensemble methods capture exponential pandemic growth curves — while acknowledging that epidemiological forecasting requires more sophisticated models (SIR, SEIR) for true predictive validity.

In [ ]:
if 'date' in df.columns:
    global_ts2 = df.groupby('date')['confirmed'].sum().reset_index().sort_values('date')
    global_ts2['days'] = (global_ts2['date'] - global_ts2['date'].min()).dt.days
    global_ts2['days_sq']   = global_ts2['days'] ** 2
    global_ts2['days_cube'] = global_ts2['days'] ** 3
    global_ts2['log_days']  = np.log1p(global_ts2['days'])

    feat_cols = ['days', 'days_sq', 'days_cube', 'log_days']
    X = global_ts2[feat_cols]
    y = global_ts2['confirmed']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    models = {
        'Ridge Regression'  : Ridge(alpha=1.0),
        'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42)
    }

    results = {}
    print(f'{"Model":<25} {"Test R²":>10} {"Test RMSE":>15}')
    print('-' * 52)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2     = r2_score(y_test, y_pred)
        rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
        results[name] = {'model': model, 'R2': r2, 'RMSE': rmse}
        print(f'{name:<25} {r2:>10.4f} {rmse:>15,.0f}')

In [ ]:
if 'date' in df.columns and results:
    best_name  = max(results, key=lambda x: results[x]['R2'])
    best_model = results[best_name]['model']
    y_pred_all = best_model.predict(X)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(global_ts2['date'], y / 1e6, color=BLUE, linewidth=2, label='Actual')
    axes[0].plot(global_ts2['date'], y_pred_all / 1e6, color=RED, linewidth=2,
                 linestyle='--', label=f'Predicted ({best_name})')
    axes[0].set_title(f'Actual vs Predicted Cumulative Cases')
    axes[0].set_ylabel('Confirmed Cases (Millions)')
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')
    axes[0].legend(fontsize=10)
    axes[0].annotate(f'R² = {results[best_name]["R2"]:.4f}',
                     xy=(0.05, 0.90), xycoords='axes fraction', fontsize=11,
                     bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

    model_names = list(results.keys())
    r2_vals     = [results[m]['R2'] for m in model_names]
    axes[1].bar(model_names, r2_vals, color=[BLUE, GREEN, RED], edgecolor='white')
    axes[1].set_title('Model Comparison — Test R²')
    axes[1].set_ylabel('R² Score')
    axes[1].set_ylim(0, 1)
    axes[1].set_xticklabels(model_names, rotation=15, ha='right')
    for i, val in enumerate(r2_vals):
        axes[1].text(i, val + 0.01, f'{val:.4f}', ha='center', fontweight='bold')

    fig.suptitle('Predictive Modeling — COVID-19 Cumulative Cases', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Important Note on Interpretation:')
    print('High R² on cumulative series should be interpreted cautiously.')
    print('Cumulative data is autocorrelated — models learn the trend rather than')
    print('true causal structure. For genuine forecasting, epidemiological models')
    print('(SIR, SEIR) incorporating transmission dynamics are more appropriate.')

---
## Section 11: Key Insights and Conclusions

### Summary of Findings

| Dimension | Key Finding |
|-----------|-------------|
| **Global Scale** | 700M+ confirmed cases, 7M+ deaths — among the deadliest modern crises |
| **Case Concentration** | USA, India, Brazil account for the majority of global case counts |
| **CFR Variation** | CFR ranged from <0.1% to >5% — a 50x difference across countries |
| **Wave Structure** | Distinct pandemic waves visible — Alpha, Delta, Omicron drove major peaks |
| **Omicron Peak** | Highest daily case counts globally — but lower CFR due to vaccination and variant characteristics |
| **Africa Undercounting** | Low confirmed counts likely reflect testing limitations, not low spread |

---

### Critical Data Limitations

**1. Undercounting of cases**  
WHO estimates true global infections were 2–4x confirmed counts. Countries with limited testing recorded only severe cases.

**2. Inconsistent death attribution**  
Some countries counted only lab-confirmed COVID deaths; others included probable deaths — making CFR comparisons imprecise.

**3. Recovery data discontinuation**  
Many health authorities stopped tracking recoveries in 2021 as mass testing became impractical.

**4. Reporting lags**  
Weekly batch reporting by some countries created artificial spikes in daily data.

---

### What the Data Tells Us About Pandemic Preparedness

Countries that performed best shared common characteristics: early decisive action, high testing capacity, functional healthcare infrastructure, and effective public communication. The data confirms that pandemic outcomes are not random — they reflect decades of prior investment in public health systems.

The COVID-19 pandemic has left a permanent mark on global data infrastructure. The rapid development of real-time disease surveillance systems, open data standards, and cross-border data sharing frameworks represents one of the pandemic's most important legacies for data science and public health.

---

If this analysis was valuable, please upvote — it helps more data scientists find this resource.

**Connect:**
- LinkedIn: [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)
- GitHub: [github.com/hassan-ali786](https://github.com/hassan-ali786)
- Portfolio: [hassanali-portfolio.vercel.app](https://hassanali-portfolio.vercel.app/)